# 16.1 - GenAI in Healthcare: Assist, Don't Replace

**Module 16 - Industry Applications** | Netsetos GenAI Engineering

This notebook builds the safety system that wraps a clinical AI model - not the model itself, but the seven guardrails that keep a human in command of it. Each cell is a keyless, runnable rule-checker: an oversight classifier, a SOAP-note validator, a patient-facing Q&A gate, a radiology schema with critical-flag routing, a PHI de-identifier, a claim-grounding check, and a final release gate that ties them together into one ship / no-ship decision.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Reproducibility note - the lesson pins any random values so every run prints identical output.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A single-line preamble, not real setup. There are no imports or installs because every check in this notebook is pure Python rule-logic with no external dependencies and no model calls.

**How the code works - step by step**
- A lone comment noting that the lesson uses seeded random values throughout for deterministic output.

**In one line:** nothing to install - the whole notebook is keyless standard-library Python.

**What you'll see:** (no output - environment prep)

## 1 - Classify the use case: assist, don't replace

Before any model call comes a decision that has nothing to do with how good the model is: how much human control does this use case require? The answer depends only on what the use case *touches*. This cell encodes the golden rule of the whole lesson as a rule you can run.

In [ ]:
# THE GOLDEN RULE - ASSIST, DON'T REPLACE: in medicine a model is a co-pilot, never the pilot. Classify each use case by whether
# it touches a clinical decision and how autonomous it is; the control scales from a disclaimer to mandatory clinician sign-off.
def control(touches_clinical, autonomous):
    if touches_clinical and autonomous:
        return "PROHIBITED - a licensed clinician must make or approve every diagnosis and treatment"
    if touches_clinical:
        return "ASSISTIVE - a clinician reviews and signs off (human-in-the-loop is mandatory)"
    return "INFORMATIONAL - add a 'not medical advice' disclaimer; no clinical decision is made"
USE_CASES = [  # (use case, touches a clinical decision?, fully autonomous?)
    ("general symptom-information chatbot", False, False),
    ("draft a clinical note from a visit",  True,  False),
    ("suggest a triage acuity level",       True,  False),
    ("autonomously diagnose and prescribe", True,  True)]
print("Classifying medical AI use cases by the required human control:")
counts = {}
for name, clinical, auto in USE_CASES:
    c = control(clinical, auto)
    tier = c.split(" -")[0]
    counts[tier] = counts.get(tier, 0) + 1
    print("  {:<36} -> {}".format(name, c))
print()
print("Tally: {}.".format(", ".join("{} {}".format(v, k) for k, v in counts.items())))
print("The rule is not a slogan: anything that touches a clinical decision needs a clinician in the loop, and fully autonomous")
print("diagnosis or treatment is off the table. The engineering job is to build the guardrails that keep the human in command.")

# Output:
# Classifying medical AI use cases by the required human control:
#   general symptom-information chatbot  -> INFORMATIONAL - add a 'not medical advice' disclaimer; no clinical decision is made
#   draft a clinical note from a visit   -> ASSISTIVE - a clinician reviews and signs off (human-in-the-loop is mandatory)
#   suggest a triage acuity level        -> ASSISTIVE - a clinician reviews and signs off (human-in-the-loop is mandatory)
#   autonomously diagnose and prescribe  -> PROHIBITED - a licensed clinician must make or approve every diagnosis and treatment
#
# Tally: 1 INFORMATIONAL, 2 ASSISTIVE, 1 PROHIBITED.
# The rule is not a slogan: anything that touches a clinical decision needs a clinician in the loop, and fully autonomous
# diagnosis or treatment is off the table. The engineering job is to build the guardrails that keep the human in command.

**Explanation**

This is a two-input decision function, not a model - it sorts use cases by required human oversight from two booleans. The core idea is that autonomy over a clinical decision, not accuracy, sets the control tier: informational gets a disclaimer, assistive gets mandatory sign-off, and fully autonomous clinical decisions are prohibited outright.

**How the code works - step by step**
- **`control(touches_clinical, autonomous)`** - the classifier: touches-a-clinical-decision AND autonomous returns PROHIBITED; touches-a-clinical-decision alone returns ASSISTIVE (human-in-the-loop); neither returns INFORMATIONAL (disclaimer only).
- **`USE_CASES`** - four labelled examples, each a tuple of (name, touches a clinical decision?, fully autonomous?).
- **The loop** - runs each case through `control`, splits off the tier name before the dash, and tallies how many land in each tier.

**In one line:** two booleans in, a required-control tier out - assist, don't replace as a runnable rule.

**What you'll see:** The four use cases print with their tiers - the symptom chatbot is INFORMATIONAL, drafting a note and suggesting triage are ASSISTIVE, and autonomous diagnose-and-prescribe is PROHIBITED - followed by the tally '1 INFORMATIONAL, 2 ASSISTIVE, 1 PROHIBITED'.

## 2 - Validate the SOAP note structure

The most common assistive use case is documentation, and clinical notes have a standard shape: SOAP - Subjective, Objective, Assessment, Plan. This validator checks a drafted note is well-formed *before* a clinician reads it, and catches one specific dangerous pattern along the way.

In [ ]:
# THE SOAP NOTE VALIDATOR: an AI-drafted clinical note must follow the SOAP structure - Subjective, Objective, Assessment, Plan -
# and it must not invent an Assessment or Plan without Objective data to support it. Validate the STRUCTURE before a clinician reads it.
REQUIRED = ["subjective", "objective", "assessment", "plan"]
def validate_soap(note):
    present = [s for s in REQUIRED if s in note and note[s].strip()]
    missing = [s for s in REQUIRED if s not in present]
    # a fabrication guard: an Assessment/Plan with no Objective findings is a red flag
    unsupported = bool(note.get("assessment") or note.get("plan")) and not note.get("objective", "").strip()
    return present, missing, unsupported
note = {"subjective": "Patient reports a sore throat for 3 days.",
        "objective": "Temp 37.9C, pharyngeal erythema, no exudate.",
        "assessment": "Likely viral pharyngitis.",
        "plan": "Supportive care; return if worse."}
present, missing, unsupported = validate_soap(note)
print("SOAP structure check ({} required sections):".format(len(REQUIRED)))
for s in REQUIRED:
    print("  [{}] {}".format("x" if s in present else " ", s.capitalize()))
print()
ok = not missing and not unsupported
print("verdict: {}".format("VALID - ready for clinician review" if ok else "INVALID"))
if missing: print("   - missing section(s): {}".format(", ".join(missing)))
if unsupported: print("   - Assessment/Plan present with no Objective data - possible fabrication, block for review")
print("A clinician still reads and signs every note. The validator just catches the machine's structural mistakes first.")

# Output:
# SOAP structure check (4 required sections):
#   [x] Subjective
#   [x] Objective
#   [x] Assessment
#   [x] Plan
#
# verdict: VALID - ready for clinician review
# A clinician still reads and signs every note. The validator just catches the machine's structural mistakes first.

**Explanation**

This is a structural validator with a fabrication guard baked in - it checks completeness and one correctness pattern, but never clinical correctness. The key idea is that an Assessment or Plan with no Objective data behind it is exactly what a hallucinating model produces, so the empty-Objective case is treated as a fabrication red flag, not just a missing field.

**How the code works - step by step**
- **`REQUIRED`** - the four SOAP sections every note must carry.
- **`validate_soap(note)`** - computes `present` (required sections that exist and are non-empty), `missing` (the rest), and `unsupported` - a boolean that fires when an Assessment or Plan exists but Objective is blank.
- **`note`** - a complete worked example: a sore-throat visit with all four sections filled.
- **The readout** - prints an `[x]`/`[ ]` checklist per section, then a verdict that is VALID only when nothing is missing and the fabrication guard has not fired.

**In one line:** all four sections present and no conclusion-without-findings -> ready for clinician review.

**What you'll see:** A four-row checklist with every SOAP section marked `[x]`, then 'verdict: VALID - ready for clinician review' and the reminder that a clinician still signs every note - the validator only catches structural mistakes first.

## 3 - Gate the patient-facing Q&A before it answers

Now a model talking directly to a patient. Safety cannot live in the answer, because by the time the model has answered the unsafe advice already exists - so the guardrail runs *before* the model responds. It makes one of three decisions on every question.

In [ ]:
# THE MEDICAL Q&A GUARDRAIL GATE: a patient-facing model must never give individualized diagnosis or dosing, must escalate an
# emergency, must add a disclaimer, and must cite a source. Run every question through the gate before the answer goes out.
EMERGENCY = {"chest pain", "trouble breathing", "suicidal", "stroke", "severe bleeding"}
INDIVIDUAL_ADVICE = {"what dose", "should i take", "do i have", "diagnose me", "how much of"}
def guardrail(question):
    q = question.lower()
    if any(e in q for e in EMERGENCY):
        return "ESCALATE - emergency signs: tell the user to call emergency services now"
    if any(a in q for a in INDIVIDUAL_ADVICE):
        return "REFUSE + REDIRECT - no individual diagnosis/dosing; advise seeing a clinician"
    return "ANSWER - general info only, with a citation and a 'not medical advice' disclaimer"
QUESTIONS = [
    "What is a normal resting heart rate?",
    "What dose of ibuprofen should I take for my back?",
    "I am having chest pain and trouble breathing, what do I do?"]
print("Guardrail decisions for three patient questions:")
for q in QUESTIONS:
    print("  {:<52} -> {}".format('"' + q[:48] + ('..."' if len(q) > 48 else '"'), guardrail(q).split(" -")[0]))
print()
for q in QUESTIONS:
    print("- {}\n    {}".format(q, guardrail(q)))
print("The gate runs BEFORE the model answers. General education is fine; individualized medical advice and emergencies are not")
print("things a chatbot handles - it redirects to a clinician or to emergency services, every time, with no exceptions.")

# Output:
# Guardrail decisions for three patient questions:
#   "What is a normal resting heart rate?"               -> ANSWER
#   "What dose of ibuprofen should I take for my back..." -> REFUSE + REDIRECT
#   "I am having chest pain and trouble breathing, wh..." -> ESCALATE
#
# - What is a normal resting heart rate?
#     ANSWER - general info only, with a citation and a 'not medical advice' disclaimer
# - What dose of ibuprofen should I take for my back?
#     REFUSE + REDIRECT - no individual diagnosis/dosing; advise seeing a clinician
# - I am having chest pain and trouble breathing, what do I do?
#     ESCALATE - emergency signs: tell the user to call emergency services now
# The gate runs BEFORE the model answers. General education is fine; individualized medical advice and emergencies are not
# things a chatbot handles - it redirects to a clinician or to emergency services, every time, with no exceptions.

**Explanation**

This is a pre-response router that classifies a question by keyword before any answer is generated. The core idea is conservatism: general education is answered with a disclaimer, but anything asking for individual dosing or diagnosis is refused-and-redirected and anything with emergency signs is escalated - the model never gets to handle the two dangerous categories.

**How the code works - step by step**
- **`EMERGENCY`** - a set of emergency-sign phrases (chest pain, trouble breathing, suicidal, stroke, severe bleeding).
- **`INDIVIDUAL_ADVICE`** - phrases that signal a request for personal dosing or diagnosis (what dose, should i take, do i have, diagnose me, how much of).
- **`guardrail(question)`** - lowercases the question and checks emergency first (ESCALATE), then individual-advice (REFUSE + REDIRECT), else ANSWER with a citation and disclaimer.
- **`QUESTIONS`** - three examples designed to hit each branch.
- **The two loops** - the first prints the short decision per question, the second prints the full decision text.

**In one line:** emergency -> escalate, individual advice -> refuse, everything else -> answer with a disclaimer.

**What you'll see:** The three questions route cleanly: 'normal resting heart rate' -> ANSWER, 'what dose of ibuprofen' -> REFUSE + REDIRECT, and 'chest pain and trouble breathing' -> ESCALATE, each shown with its full decision text below.

## 4 - Validate the radiology report and route on the critical flag

Radiology gives structured output a life-or-death edge. A report has required fields, and one of them - the critical flag - is not documentation, it is *routing*: it is what pages a radiologist for an urgent read. This validator checks the schema and that the flag matches the findings.

In [ ]:
# THE RADIOLOGY STRUCTURED-REPORT SCHEMA: free-text findings become a structured report with required fields, and any CRITICAL
# finding is flagged for an urgent read. Validate the schema and the critical-flag routing before the report reaches the queue.
SCHEMA = ["modality", "findings", "impression", "recommendation", "critical"]
CRITICAL_TERMS = {"pneumothorax", "hemorrhage", "aortic dissection", "free air", "mass effect"}
def validate_report(r):
    missing = [f for f in SCHEMA if f not in r]
    text = (r.get("findings", "") + " " + r.get("impression", "")).lower()
    should_flag = any(t in text for t in CRITICAL_TERMS)
    flag_ok = r.get("critical", False) == should_flag
    return missing, should_flag, flag_ok
report = {"modality": "chest CT",
          "findings": "Large right-sided pneumothorax with mediastinal shift.",
          "impression": "Tension pneumothorax.",
          "recommendation": "Immediate clinical correlation and decompression.",
          "critical": True}
missing, should_flag, flag_ok = validate_report(report)
print("Radiology report schema check ({} required fields):".format(len(SCHEMA)))
for f in SCHEMA:
    print("  [{}] {}".format("x" if f in report else " ", f))
print()
print("critical finding detected: {}  |  report flagged critical: {}  |  routing {}".format(
    should_flag, report.get("critical", False), "OK" if flag_ok else "MISMATCH - fix before queueing"))
print("verdict: {}".format("VALID - route to urgent read" if not missing and flag_ok and should_flag else "check the report"))
print("Structured output is not just tidy - the 'critical' field is what pages a radiologist in minutes instead of hours.")

# Output:
# Radiology report schema check (5 required fields):
#   [x] modality
#   [x] findings
#   [x] impression
#   [x] recommendation
#   [x] critical
#
# critical finding detected: True  |  report flagged critical: True  |  routing OK
# verdict: VALID - route to urgent read
# Structured output is not just tidy - the 'critical' field is what pages a radiologist in minutes instead of hours.

**Explanation**

This is a schema validator that also cross-checks a routing field against the report's own text. The distinguishing idea is that the critical flag must *agree with* the findings: a well-formed report with a genuine critical finding but the flag left off is a routing bug, not a formatting slip, because that flag is what triggers the urgent read.

**How the code works - step by step**
- **`SCHEMA`** - the five required fields: modality, findings, impression, recommendation, critical.
- **`CRITICAL_TERMS`** - findings that must set the critical flag (pneumothorax, hemorrhage, aortic dissection, free air, mass effect).
- **`validate_report(r)`** - computes `missing` fields, scans findings+impression for a critical term to decide `should_flag`, and `flag_ok` - whether the report's `critical` value matches `should_flag`.
- **`report`** - a complete chest-CT example with a tension pneumothorax and `critical: True`.
- **The readout** - prints the field checklist, then the detected-vs-flagged comparison and a routing OK/MISMATCH verdict.

**In one line:** all fields present and the critical flag matches the findings -> route to urgent read.

**What you'll see:** A five-field checklist all marked `[x]`, then 'critical finding detected: True | report flagged critical: True | routing OK' and 'verdict: VALID - route to urgent read'.

## 5 - De-identify PHI before it crosses any boundary

Every system in this lesson shares one boundary: the moment clinical text leaves your control to reach a hosted model or a log. Before it crosses, the protected health information must be stripped. This cell applies Lesson 15.4's PII scrubbing to the clinical setting, where the identifiers have a legal definition.

In [ ]:
# PHI DE-IDENTIFICATION (HIPAA Safe Harbor): before any text goes to a model or a log, strip the protected health information -
# names, record numbers, dates, contacts. You cannot honour privacy after a leak; scrub at the boundary. (Ties to Lesson 15.4.)
PHI_FIELDS = {"name": "[NAME]", "mrn": "[MRN]", "dob": "[DATE]", "phone": "[PHONE]",
              "address": "[ADDRESS]", "email": "[EMAIL]", "ssn": "[SSN]"}
record = {"name": "Jane Doe", "mrn": "MRN-4471902", "dob": "1984-02-11", "phone": "555-0142",
          "address": "12 Oak St", "email": "jane@x.com", "ssn": "123-45-6789",
          "note": "Patient with sore throat, no fever. Follow up in one week."}
scrubbed = dict(record)
removed = []
for field, token in PHI_FIELDS.items():
    if field in scrubbed and scrubbed[field]:
        scrubbed[field] = token
        removed.append(field)
print("PHI de-identification (HIPAA Safe Harbor - 18 identifier categories):")
print("  identifiers removed: {} -> {}".format(len(removed), ", ".join(removed)))
print("  clinical note (retained, no PHI): {}".format(scrubbed["note"]))
print()
print("{} of the record's identifier fields were replaced with tokens before the note left the boundary.".format(len(removed)))
print("De-identify at ingestion, not after: a model call or a log line that carries raw PHI is a breach you cannot take back.")

# Output:
# PHI de-identification (HIPAA Safe Harbor - 18 identifier categories):
#   identifiers removed: 7 -> name, mrn, dob, phone, address, email, ssn
#   clinical note (retained, no PHI): Patient with sore throat, no fever. Follow up in one week.
#
# 7 of the record's identifier fields were replaced with tokens before the note left the boundary.
# De-identify at ingestion, not after: a model call or a log line that carries raw PHI is a breach you cannot take back.

**Explanation**

This is a boundary-time field scrubber that replaces identifier fields with tokens while keeping the clinical signal. The core idea is de-identify-at-ingestion, not after: once raw PHI has been sent in a model call or written to a log the breach cannot be undone, so the note survives but the seven identifier fields are tokenized before anything leaves.

**How the code works - step by step**
- **`PHI_FIELDS`** - a map from identifier field names to their replacement tokens ([NAME], [MRN], [DATE], and so on).
- **`record`** - a patient record carrying seven identifiers plus a `note` field that holds no PHI.
- **The scrub loop** - copies the record, and for each PHI field that exists and is non-empty, overwrites it with its token and appends the field name to `removed`.
- **The readout** - prints how many identifiers were removed and the retained clinical note.

**In one line:** tokenize the identifier fields, keep the note - scrub at the boundary because a leak cannot be undone.

**What you'll see:** Prints 'identifiers removed: 7 -> name, mrn, dob, phone, address, email, ssn', the retained clinical note about the sore throat, and confirmation that 7 fields were tokenized before the note left the boundary.

## 6 - Ground every clinical claim against the source

A de-identified, well-structured output can still be *false*, and this is the failure that most defines clinical AI risk. The dangerous case is not a garbled sentence - it is a fluent, confident, perfectly formatted claim about a medication the source never contained. This check traces each claim back to the source.

In [ ]:
# THE GROUNDING / HALLUCINATION CHECK: every clinical claim in an AI summary must be supported by the source note. Check each
# claim's key terms against the source; an unsupported claim (a drug the note never mentions) is a hallucination - block it.
source = ("Patient reports sore throat for three days. Exam: temp 37.9C, pharyngeal erythema, no exudate. "
          "Assessment: viral pharyngitis. Plan: supportive care, fluids, rest.")
claims = ["The patient has a sore throat.",
          "The exam showed pharyngeal erythema.",
          "The patient was prescribed amoxicillin."]      # the last claim is NOT in the source
def supported(claim, src):
    stop = {"the", "a", "an", "was", "has", "is", "of", "for", "patient"}
    key = [w.strip(".,").lower() for w in claim.split() if w.strip(".,").lower() not in stop]
    hits = sum(1 for w in key if w in src.lower())
    return hits >= max(1, len(key) - 1)                   # nearly all key terms must appear in the source
print("Grounding check - is each claim supported by the source note?")
unsupported = []
for c in claims:
    ok = supported(c, source)
    if not ok: unsupported.append(c)
    print("  [{}] {}".format("grounded" if ok else "UNSUPPORTED", c))
print()
print("{} unsupported claim(s) -> block the summary and flag for review.".format(len(unsupported)))
for c in unsupported:
    print("   - '{}' - no support in the source (a fabricated medication is a patient-safety event)".format(c))
print("A fluent summary is not a correct one. Ground every clinical claim to the source, or a hallucinated drug reaches a chart.")

# Output:
# Grounding check - is each claim supported by the source note?
#   [grounded] The patient has a sore throat.
#   [grounded] The exam showed pharyngeal erythema.
#   [UNSUPPORTED] The patient was prescribed amoxicillin.
#
# 1 unsupported claim(s) -> block the summary and flag for review.
#    - 'The patient was prescribed amoxicillin.' - no support in the source (a fabricated medication is a patient-safety event)
# A fluent summary is not a correct one. Ground every clinical claim to the source, or a hallucinated drug reaches a chart.

**Explanation**

This is a claim-verification harness that tests support against the source text, not fluency. The core idea is trust-by-source instead of trust-by-fluency: it decomposes each claim into key terms and requires nearly all of them to appear in the source, so a fabricated medication with no basis in the note is caught and blocked.

**How the code works - step by step**
- **`source`** - the ground-truth note the summary must trace back to.
- **`claims`** - three summary claims; the first two are in the source, the third (prescribed amoxicillin) is not.
- **`supported(claim, src)`** - drops stopwords, extracts key terms, counts how many appear in the source, and returns True only when nearly all key terms match (`hits >= len(key) - 1`).
- **The loop** - grades each claim grounded or UNSUPPORTED and collects the unsupported ones.
- **The readout** - reports the count of unsupported claims and blocks the summary for review.

**In one line:** decompose each claim to key terms, require them in the source, block anything unsupported.

**What you'll see:** The first two claims print as [grounded], the amoxicillin claim as [UNSUPPORTED], then '1 unsupported claim(s) -> block the summary and flag for review' naming the fabricated medication as a patient-safety event.

## 7 - Log the audit trail and gate the release

Everything in this lesson converges on one moment: the decision to release a medical AI output. The release gate is the checklist that decision must pass, and its defining property is that it can say no. This cell ties de-identification, grounding, guardrails, sign-off, and audit logging into one ship / no-ship call.

In [ ]:
# THE AUDIT TRAIL + DEPLOYMENT GATE: every medical AI output is logged (model version, timestamp, a content hash, the clinician
# sign-off) and gated before release. The gate ties the whole lesson together: de-identified, grounded, guardrailed, signed, logged.
def content_hash(text):                                   # a toy deterministic hash (a real system uses SHA-256)
    return sum(ord(c) for c in text) % 100000
audit = {"model": "med-llm@2026-03-01", "input_hash": content_hash("visit transcript"),
         "output_hash": content_hash("SOAP note"), "clinician_signoff": True}
GATE = {                                                  # each must hold to release a medical AI output
    "PHI de-identified before the model call": True,
    "output grounded to the source (no hallucination)": True,
    "safety guardrails passed": True,
    "clinician reviewed and signed off": True,
    "audit trail written (model, hashes, timestamp)": False}   # UNMET - logging not wired up
unmet = [k for k, ok in GATE.items() if not ok]
print("Audit record: model={} input_hash={} output_hash={} signoff={}".format(
    audit["model"], audit["input_hash"], audit["output_hash"], audit["clinician_signoff"]))
print()
print("MEDICAL AI RELEASE GATE:")
for k, ok in GATE.items():
    print("  [{}] {}".format("x" if ok else " ", k))
print()
print("decision: {}".format("RELEASE" if not unmet else "BLOCK - {} unmet:".format(len(unmet))))
for u in unmet:
    print("   - " + u)
print("The audit trail is not bureaucracy: when an AI output is questioned, you must show exactly what model produced what, from")
print("what input, and which clinician signed it. De-identify, ground, guardrail, sign, and log - or it does not ship.")

# Output:
# Audit record: model=med-llm@2026-03-01 input_hash=1689 output_hash=777 signoff=True
#
# MEDICAL AI RELEASE GATE:
#   [x] PHI de-identified before the model call
#   [x] output grounded to the source (no hallucination)
#   [x] safety guardrails passed
#   [x] clinician reviewed and signed off
#   [ ] audit trail written (model, hashes, timestamp)
#
# decision: BLOCK - 1 unmet:
#    - audit trail written (model, hashes, timestamp)
# The audit trail is not bureaucracy: when an AI output is questioned, you must show exactly what model produced what, from
# what input, and which clinician signed it. De-identify, ground, guardrail, sign, and log - or it does not ship.

**Explanation**

This is the integrative gate - a boolean checklist plus an audit record that produces a single RELEASE or BLOCK decision. The key idea is that any one unmet item blocks the release: here four checks pass but the audit trail is not wired up, so the output does not ship, because in a clinical setting an output you cannot trace is one you cannot defend.

**How the code works - step by step**
- **`content_hash(text)`** - a toy deterministic hash (a real system uses SHA-256) to stand in for content fingerprints.
- **`audit`** - the audit record: model version, input and output hashes, and the clinician sign-off flag.
- **`GATE`** - the five release conditions (de-identified, grounded, guardrails passed, clinician signed, audit trail written) with the audit trail deliberately set False.
- **`unmet`** - the list of failing conditions.
- **The readout** - prints the audit record, the gate checklist, and a decision that is RELEASE only when `unmet` is empty - otherwise BLOCK with the unmet items listed.

**In one line:** de-identify, ground, guardrail, sign, and log - one unmet item forces a BLOCK.

**What you'll see:** Prints the audit record, a five-item gate checklist with the audit-trail row unchecked, then 'decision: BLOCK - 1 unmet:' naming the missing audit trail - four passing checks are not enough to ship.

Across seven checks you built a medical AI system as a pipeline of guardrails rather than a smarter model: classify the use case for the oversight it needs, validate note structure, gate the patient-facing Q&A before it answers, route on the radiology critical flag, strip PHI at the boundary, ground every claim to its source, and gate the release on de-identification, grounding, guardrails, sign-off, and an audit trail. The one sentence to carry forward is that the model drafts, the human decides, and the guardrails are what make the draft safe to touch a chart. Lessons 16.4 and 16.5 reuse this exact pattern at a different intensity - the test suite and CI gate for AI-written code, and the guardrails behind no-code building.